# ClauseFinder — DELTA embed (recovered specs)

Fixes the header-parsing bug that silently skipped 290 specs (incl. TS 24.501).
Re-chunks with the fixed chunker, embeds ONLY the 69k new chunks, appends to the
existing index, rebuilds FTS, replaces edges, snapshots to HF continuously.
GPU: T4 is plenty (~40 min). Run All.

In [ ]:
!pip -q install sentence-transformers lancedb typer huggingface_hub pydantic

In [ ]:
from huggingface_hub import HfApi, login

login()
api = HfApi()
REPO = f"{HfApi().whoami()['name']}/clausefinder-index"  # your own namespace
print("login ok")

In [ ]:
# pull existing snapshot: full index + old parsed (for delta computation)
from huggingface_hub import snapshot_download

snapshot_download(REPO, repo_type="dataset", local_dir=".", allow_patterns=["data/index/**", "data/parsed/**"])
ck = open("data/index/.embed_checkpoint").read() if __import__("os").path.exists("data/index/.embed_checkpoint") else "?"
print("snapshot pulled; full-run checkpoint:", ck)

In [ ]:
%%writefile download_gsma.py
"""Download spec markdown from the official GSMA/3GPP mirror on Hugging Face.

Layout: marked/Rel-XX/{NN}_series/{spec}/raw.md  (images alongside, skipped here).
"""
from __future__ import annotations

from pathlib import Path

import typer
from huggingface_hub import snapshot_download

DATASET = "GSMA/3GPP"
DEFAULT_SERIES = [23, 24, 29, 33, 38]
DEFAULT_RELEASES = ["Rel-17", "Rel-18"]

app = typer.Typer(add_completion=False)


@app.command()
def download(
    out: Path = typer.Option(Path("data/corpus"), help="Local corpus root"),
    series: list[int] = typer.Option(DEFAULT_SERIES, help="3GPP series numbers"),
    releases: list[str] = typer.Option(DEFAULT_RELEASES, help="Releases, e.g. Rel-18"),
    spec: list[str] = typer.Option(None, help="Restrict to specific specs, e.g. 38331"),
) -> None:
    patterns = []
    for rel in releases:
        for s in series:
            if spec:
                patterns += [f"marked/{rel}/{s}_series/{sp}/raw.md" for sp in spec]
            else:
                patterns.append(f"marked/{rel}/{s}_series/*/raw.md")
    path = snapshot_download(
        repo_id=DATASET,
        repo_type="dataset",
        allow_patterns=patterns,
        local_dir=out,
    )
    n = sum(1 for _ in Path(path).rglob("raw.md"))
    typer.echo(f"downloaded {n} spec files under {path}")


if __name__ == "__main__":
    app()


In [ ]:
%%writefile chunker.py
"""Clause-aware chunking of GSMA/3GPP markdown.

Clause numbers live in heading TEXT; markdown heading levels are inconsistent
in the GSMA conversion (e.g. `## 5.2.2` next to `### 4.2.1`), so hierarchy is
derived from the clause number itself, never from `#` depth.

Output: parent units (one per clause, capped) and child chunks (retrieval
units, 200-350 tokens, breadcrumb-prefixed) plus cross-reference edges.
"""

import hashlib
import json
import re
from dataclasses import dataclass, field
from pathlib import Path

from pydantic import BaseModel

# `## 5.2.2.3.3a Title` / `## A.1 Title` / `## Annex B (informative): Title`
CLAUSE_RE = re.compile(r"^#{1,6}\s+\**((?:\d+|[A-Z])(?:\.\d+)*[a-z]?)\**\s+(.+?)\s*$")
ANNEX_RE = re.compile(r"^#{1,6}\s+\**Annex\s+([A-Z])\**\s*(?:\((normative|informative)\))?:?\s*(.*?)\s*$", re.I)
# GSMA markdown has two header shapes:
#   (a) "# 3GPP TS 38.331 V18.5.0"          — single heading line
#   (b) "**3GPP TS**\n\n**24.501 V18.5.0**" — split bold lines (~24% of specs)
TITLE_RE = re.compile(
    r"(?:#\s+|\*\*)3GPP\s+(TS|TR)(?:\*\*)?\s+(?:\*\*)?([\d.\-]+)\s+V([\d.]+)",
    re.DOTALL)
XREF_CLAUSE_RE = re.compile(r"\bclauses?\s+((?:\d+|[A-Z])(?:\.\d+)+[a-z]?)", re.I)
XREF_SPEC_RE = re.compile(r"\bTS\s?(\d{2}\.\d{3}(?:-\d+)?)")

SKIP_TITLES = {"contents", "foreword", "references"}
CHILD_TARGET = 300   # tokens (approx)
CHILD_MAX = 350
PARENT_MAX = 1200
EMBED_MAX = 1400     # OTel-568M fine-tune length guard


def ntokens(text: str) -> int:
    """Cheap token estimate (~1.3 tokens/word for technical text)."""
    return int(len(text.split()) * 1.3)


class Chunk(BaseModel):
    id: str
    parent_id: str | None = None
    spec: str            # "TS 38.331"
    series: int
    release: str         # "Rel-18"
    version: str         # "18.0.0"
    clause: str          # "5.2.2.1" or "A.1"
    clause_title: str
    breadcrumb: str
    text: str
    is_table: bool = False
    annex: bool = False
    normative: bool = True
    over_embed_limit: bool = False
    url: str = ""


class XrefEdge(BaseModel):
    src_spec: str
    src_clause: str
    dst_spec: str        # same spec unless a TS reference is nearby
    dst_clause: str


@dataclass
class _Section:
    clause: str
    title: str
    lines: list[str] = field(default_factory=list)
    annex: bool = False
    normative: bool = True


def _clause_rank(clause: str) -> list[str]:
    return clause.split(".")


def _spec_url(spec_digits: str) -> str:
    # Deep link to the official 3GPP spec page.
    return f"https://www.3gpp.org/DynaReport/{spec_digits.replace('.', '').replace('-', '')}.htm"


def parse_spec(md_path: Path, release: str, series: int) -> tuple[list[Chunk], list[XrefEdge], dict]:
    text = md_path.read_text(errors="replace")
    lines = text.splitlines()

    # Identity comes from the GSMA path (authoritative): .../<series>_series/<digits>/raw.md
    # Header formats vary too much to trust (single-line, split-bold, and
    # broken Word-field conversions all occur); it only contributes version.
    raw_digits = md_path.parent.name          # e.g. "24501" or "38523-3"
    base, _, suffix = raw_digits.partition("-")
    spec_digits = f"{base[:2]}.{base[2:]}" + (f"-{suffix}" if suffix else "")
    m = TITLE_RE.search(text[:2000])
    if m:
        doc_type, _, version = m.groups()
    else:
        doc_type = "TR" if "Technical Report" in text[:3000] else "TS"
        vm = re.search(r"\bV(\d+\.\d+\.\d+)", text[:5000])
        version = vm.group(1) if vm else "unknown"
    spec = f"{doc_type} {spec_digits}"
    url = _spec_url(spec_digits)

    # --- split into clause sections -------------------------------------
    sections: list[_Section] = []
    cur: _Section | None = None
    annex_ctx: tuple[str, bool] | None = None  # (letter, normative)

    for ln in lines:
        am = ANNEX_RE.match(ln)
        cm = CLAUSE_RE.match(ln) if not am else None
        if am:
            letter, kind, title = am.groups()
            annex_ctx = (letter, (kind or "").lower() != "informative")
            cur = _Section(letter, title or f"Annex {letter}", annex=True, normative=annex_ctx[1])
            sections.append(cur)
            continue
        if cm:
            clause, title = cm.group(1), cm.group(2).replace("*", "").strip()
            is_annex = bool(annex_ctx) and clause[0].isalpha()
            normative = annex_ctx[1] if is_annex and annex_ctx else True
            if not clause[0].isalpha():
                annex_ctx = None
                is_annex, normative = False, True
            cur = _Section(clause, title, annex=is_annex, normative=normative)
            sections.append(cur)
            continue
        if cur is not None:
            cur.lines.append(ln)

    # --- build chunks ----------------------------------------------------
    chunks: list[Chunk] = []
    edges: list[XrefEdge] = []
    over_limit = 0
    path: list[tuple[str, str]] = []  # (clause, title) stack for breadcrumbs

    for sec in sections:
        # ancestors are strict clause-number prefixes (5.3.5.3 ⇒ 5 > 5.3 > 5.3.5);
        # maintain the stack for EVERY section — container clauses often have no body
        while path and not (sec.clause + ".").startswith(path[-1][0] + "."):
            path.pop()
        crumb_mid = " > ".join(f"{c} {t}" for c, t in path)
        path.append((sec.clause, sec.title))

        if sec.title.strip().lower() in SKIP_TITLES:
            continue
        body = "\n".join(sec.lines).strip()
        if not body:
            continue

        breadcrumb = f"{spec} {release} — " + (f"{crumb_mid} > " if crumb_mid else "") + f"{sec.clause} {sec.title}"
        pid = hashlib.sha1(f"{spec}|{release}|{sec.clause}".encode()).hexdigest()[:16]

        # xref edges from this clause's body
        for xm in XREF_CLAUSE_RE.finditer(body):
            ctx = body[max(0, xm.start() - 60):xm.start()]
            sm = XREF_SPEC_RE.search(ctx)
            edges.append(XrefEdge(
                src_spec=spec, src_clause=sec.clause,
                dst_spec=f"TS {sm.group(1)}" if sm else spec,
                dst_clause=xm.group(1),
            ))

        common = dict(
            parent_id=pid, spec=spec, series=series, release=release,
            version=version, clause=sec.clause, clause_title=sec.title,
            breadcrumb=breadcrumb, annex=sec.annex, normative=sec.normative, url=url,
        )

        for i, (piece, is_table) in enumerate(_split_children(body)):
            over = ntokens(breadcrumb + piece) > EMBED_MAX
            over_limit += over
            chunks.append(Chunk(id=f"{pid}-{i}", text=piece, is_table=is_table,
                                over_embed_limit=over, **common))

    stats = {
        "spec": spec, "release": release, "version": version,
        "clauses": len(sections), "chunks": len(chunks),
        "edges": len(edges), "over_embed_limit": over_limit,
    }
    return chunks, edges, stats


def _split_children(body: str) -> list[tuple[str, bool]]:
    """Split clause body into 200-350-token pieces; tables stay atomic."""
    blocks: list[tuple[str, bool]] = []   # (block, is_table)
    cur_lines: list[str] = []
    in_table = False
    for ln in body.splitlines():
        is_t = ln.lstrip().startswith("|")
        if is_t != in_table and cur_lines:
            blocks.append(("\n".join(cur_lines).strip(), in_table))
            cur_lines = []
        in_table = is_t
        cur_lines.append(ln)
    if cur_lines:
        blocks.append(("\n".join(cur_lines).strip(), in_table))

    out: list[tuple[str, bool]] = []
    buf, buf_tok = [], 0
    for block, is_table in blocks:
        if not block:
            continue
        if is_table:
            if buf:
                out.append(("\n".join(buf), False))
                buf, buf_tok = [], 0
            out.append((block, True))
            continue
        for para in re.split(r"\n{2,}", block):
            t = ntokens(para)
            if buf_tok + t > CHILD_MAX and buf:
                out.append(("\n".join(buf), False))
                buf, buf_tok = [], 0
            buf.append(para)
            buf_tok += t
            if buf_tok >= CHILD_TARGET:
                out.append(("\n".join(buf), False))
                buf, buf_tok = [], 0
    if buf:
        out.append(("\n".join(buf), False))
    return [(p, t) for p, t in out if p.strip()]


def parse_corpus(corpus_root: Path, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    all_stats = []
    with (out_dir / "chunks.jsonl").open("w") as cf, (out_dir / "edges.jsonl").open("w") as ef:
        for md in sorted(corpus_root.rglob("raw.md")):
            release = next(p for p in md.parts if p.startswith("Rel-"))
            series = int(next(p for p in md.parts if p.endswith("_series")).split("_")[0])
            try:
                chunks, edges, stats = parse_spec(md, release, series)
            except ValueError as e:
                print(f"SKIP {md}: {e}")
                continue
            for c in chunks:
                cf.write(c.model_dump_json() + "\n")
            for e in edges:
                ef.write(e.model_dump_json() + "\n")
            all_stats.append(stats)
            print(f"{stats['spec']} {release}: {stats['chunks']} chunks, "
                  f"{stats['edges']} edges, {stats['over_embed_limit']} over-limit")
    (out_dir / "manifest.json").write_text(json.dumps(all_stats, indent=1))


if __name__ == "__main__":
    import typer
    typer.run(lambda corpus: parse_corpus(Path(corpus), Path("data/parsed")))


In [ ]:
%%writefile embed_index.py
"""Embed child chunks with OTel-Embedding-568M and build the LanceDB index.

Streaming + checkpointed: chunks are read lazily, embedded in small batches,
and appended to LanceDB immediately — peak RAM is one batch, not the corpus.
A checkpoint file records how many rows are committed; rerunning resumes
where the last run stopped (crash/OOM/restart safe).

No ANN index is created on purpose: at this corpus size, LanceDB brute-force
(flat) search is exact — the literature (Telco-RAG) measured HNSW losing
accuracy vs flat on 3GPP retrieval. FTS (tantivy BM25) rides the same table.
"""

import json
from itertools import islice
from pathlib import Path

import lancedb
import typer

MODEL_ID = "farbodtavakkoli/OTel-Embedding-568M"
EMBED_MAX_TOKENS = 1400  # OTel fine-tune length; inputs truncated by tokenizer

app = typer.Typer(add_completion=False)


def load_model():
    import torch
    from sentence_transformers import SentenceTransformer
    device = ("cuda" if torch.cuda.is_available()
              else "mps" if torch.backends.mps.is_available() else "cpu")
    kwargs = {}
    if device == "cuda":
        # fp16 halves VRAM (T4 14.5GB OOMs at fp32 batch 64 × 1400 tok);
        # retrieval quality unaffected. MPS/CPU stay fp32.
        kwargs["model_kwargs"] = {"torch_dtype": torch.float16}
    model = SentenceTransformer(MODEL_ID, trust_remote_code=True, device=device, **kwargs)
    model.max_seq_length = EMBED_MAX_TOKENS
    typer.echo(f"embedding on device: {device} ({'fp16' if device == 'cuda' else 'fp32'})")
    return model


def _commit(db, tbl, pending: list[dict], done: int, ckpt: Path) -> int:
    if tbl is None and "chunks" not in db.table_names():
        db.create_table("chunks", pending)
    else:
        db.open_table("chunks").add(pending)
    done += len(pending)
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    ckpt.write_text(str(done))
    return done


def _batches(path: Path, size: int, skip: int):
    with path.open() as f:
        it = (json.loads(l) for l in islice(f, skip, None))
        while chunk := list(islice(it, size)):
            yield chunk


@app.command()
def build(
    parsed: Path = typer.Option(Path("data/parsed"), help="chunker output dir"),
    db_path: Path = typer.Option(Path("data/index"), help="LanceDB dir"),
    batch: int = typer.Option(16, help="embed batch (peak-RAM knob)"),
    commit_every: int = typer.Option(512, help="rows per LanceDB append"),
    fresh: bool = typer.Option(False, help="drop table + checkpoint, start over"),
) -> None:
    # Checkpoint is scoped to the parsed dir: a delta run (different chunks
    # file, different ordering) must never resume against the full-run offset.
    ckpt = db_path / f".embed_checkpoint_{parsed.name}"
    total = sum(1 for _ in (parsed / "chunks.jsonl").open())
    db = lancedb.connect(db_path)

    if fresh and "chunks" in db.table_names():
        db.drop_table("chunks")
        ckpt.unlink(missing_ok=True)
    done = int(ckpt.read_text()) if ckpt.exists() and "chunks" in db.table_names() else 0
    if done >= total:
        typer.echo(f"already complete ({done}/{total}); finalizing indexes only")
    else:
        typer.echo(f"embedding {total - done}/{total} chunks with {MODEL_ID} "
                   f"(resume at {done}) …")
        model = load_model()
        tbl = db.open_table("chunks") if "chunks" in db.table_names() else None
        pending: list[dict] = []
        for rows in _batches(parsed / "chunks.jsonl", batch, done):
            texts = [r["breadcrumb"] + "\n" + r["text"] for r in rows]
            vecs = model.encode(texts, batch_size=batch,
                                normalize_embeddings=True)
            for r, v in zip(rows, vecs):
                r["vector"] = v.tolist()
            pending.extend(rows)
            if len(pending) >= commit_every:
                done = _commit(db, tbl := (tbl if tbl is not None else None), pending, done, ckpt)
                tbl = db.open_table("chunks")
                typer.echo(f"  committed {done}/{total}")
                pending = []
        if pending:
            done = _commit(db, tbl, pending, done, ckpt)

    tbl = db.open_table("chunks")
    typer.echo("building FTS + scalar indexes …")
    tbl.create_fts_index("text", replace=True)
    tbl.create_scalar_index("release", replace=True)
    tbl.create_scalar_index("series", replace=True)

    edges = [json.loads(l) for l in (parsed / "edges.jsonl").open()]
    if edges:
        if "edges" in db.table_names():
            db.drop_table("edges")
        db.create_table("edges", edges)
    typer.echo(f"index built at {db_path}: {tbl.count_rows()} chunks, {len(edges)} edges")


if __name__ == "__main__":
    app()


In [ ]:
# corpus + re-chunk with FIXED chunker + compute delta
import json
import os

if not os.path.exists("data/corpus/marked"):
    !python download_gsma.py --out data/corpus
from pathlib import Path

from chunker import parse_corpus

parse_corpus(Path("data/corpus"), Path("data/parsed_v2"))
old_ids = set(json.loads(l)["id"] for l in open("data/parsed/chunks.jsonl"))
os.makedirs("data/parsed_delta", exist_ok=True)
n = d = 0
with open("data/parsed_delta/chunks.jsonl", "w") as out:
    for l in open("data/parsed_v2/chunks.jsonl"):
        n += 1
        if json.loads(l)["id"] not in old_ids:
            out.write(l); d += 1
import shutil

for f in ("edges.jsonl", "manifest.json"):
    shutil.copy(f"data/parsed_v2/{f}", f"data/parsed_delta/{f}")
shutil.copy("data/parsed/acronyms.json", "data/parsed_delta/acronyms.json")
print(f"v2 total {n} | delta {d}")

In [ ]:
# embed the delta in slices, snapshot to HF after each (reset-proof)
import subprocess
import sys

SLICE = 25600
def upload():
    api.upload_folder(folder_path="data/index", repo_id=REPO, repo_type="dataset", path_in_repo="data/index")
    api.upload_folder(folder_path="data/parsed_v2", repo_id=REPO, repo_type="dataset", path_in_repo="data/parsed_v2")
total = sum(1 for _ in open("data/parsed_delta/chunks.jsonl"))
while True:
    r = subprocess.run([sys.executable, "embed_index.py", "--parsed", "data/parsed_delta",
                        "--batch", "16", "--commit-every", "1024", "--stop-after", str(SLICE)])
    if r.returncode != 0:
        raise SystemExit(f"embed exited {r.returncode}")
    done = int(open("data/index/.embed_checkpoint_parsed_delta").read())
    print(f"snapshotting {done}/{total} …"); upload()
    if done >= total:
        print("DELTA EMBED COMPLETE — final snapshot uploaded"); break

In [ ]:
print("done — tell Claude the delta snapshot is on HF")